In [ ]:
import pandas as pd
import os
import numpy as np
import openpyxl

In [ ]:
def nfiPreprocessing(df, output_name):
    # 임상도 수종 및 코드 분류에 따라 수종코드(SID) 부여하기
    code_species_dict = {11: ['소나무'], 12:['잣나무', '섬잣나무', '눈잣나무', '스트로브잣나무'], 13: ['일본잎갈나무', '잎갈나무'], 14: ['리기다소나무', '리기테다소나무', '방크스소나무'],
                        15: ['곰솔'], 16: ['전나무', '구상나무', '분비나무'], 17: ['편백', '화백'], 18: ['삼나무', '낙우송','메타세콰이아'], 19: ['가문비나무', '독일가문비나무', '종비나무'],
                        20: ['비자나무', '개비자나무'], 21: ['은행나무'], 31: ['상수리나무'], 32: ['신갈나무'], 33: ['굴참나무'], 34: ['갈찬나무', '떡갈나무', '졸참나무'],
                        35: ['오리나무', '물오리나무', '사방오리'], 36: ['고로쇠나무'], 37: ['자작나무', '거제수나무'],  38: ['박달나무', '개박달나무', '물박달나무'], 39: ['밤나무'],
                        40: ['물푸레나무', '들메나무', '물들메나무'], 41: ['서어나무', '개서어나무'], 42: ['때죽나무', '쪽동백나무'], 43: ['호두나무', '가래나무'], 44:['백합나무'], 
                        45: ['미루나무', '은사시나무', '이태리포플러나무', '수원사시나무'], 46: ['벚나무', '양벚나무', '산벚나무', '꽃벚나무', '왕벚나무', '잔털벚나무', '개벚나무', '올벚나무', '섬벚나무', '섬개벚나무', '산개벚지나무', '개벚지나무', ''], 47: ['느티나무'],  48:['층층나무', '곰의말채나무'],
                        49: ['아까시나무'], 61: ['가시나무', '붉가시나무', '종가시나무', '참가시나무', '개가시나무'], 62: ['구실잣밤나무'], 63: ['녹나무'], 64: ['굴거리나무'], 65: ['황칠나무'], 66: ['사스레피나무'], 67: ['후박나무'],
                         68: ['새덕이', '참식나무', '생달나무']}
    id_lst = [11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 61, 62, 63, 64, 65, 66, 67, 68]
    name_lst1 = ['소나무', '잣나무', '낙엽송', '리기다소나무', '곰솔', '전나무', '편백나무', '삼나무', '가문비나무', '비자나무', '은행나무', '상수리나무', '신갈나무', '굴참나무', '기타참나무류', '오리나무', '고로쇠나무', '자작나무', '박달나무', '밤나무', '물푸레나무', '서어나무', '때죽나무', '호두나무', '백합나무', '포플러', '벚나무', 
                 '느티나무', '층층나무', '아까시나무', '가시나무', '구실잣밤나무', '녹나무', '굴거리나무', '황칠나무','사스레피나무', '후박나무','새덕이']
    name_lst2 = ['기타침엽수', '기타 참나무류', '기타활엽수']
    code_name_dict = {i: j for i, j in zip(id_lst, name_lst1)}

    # 임상도에 따라 NFI 수종명 재분류
    print(r"Reclassify based on Imsang code...")
    nfi_names = df['수종명'].unique()
    nfi_imsang = [df.loc[(df['수종명']==name), '침활구분'].unique()[0] for name in nfi_names]
    nfi_dict = {i : j for i, j in zip(nfi_names, nfi_imsang)}

    # 속성 추출 및 단위 환산
    print("Extract necessary columns & Convert Unit...")
    df2 = df[['표본점번호', '조사차기', '수종명', '침활구분', '흉고직경', '수고', '지하고', '해발고(m)', '경사(degree)', '방위각(º)','평균수관밀도(%)','좌표N', '좌표E']]
    cm_to_inch = 0.3937
    cm_to_ft = 0.0328084
    df2['흉고직경'] = df2['흉고직경'].apply(lambda x: x * cm_to_inch)
    df2['수고'] = df2['수고'].apply(lambda x: x * cm_to_ft)
    df2['지하고'] = df2['지하고'].apply(lambda x: x * cm_to_ft)
    df2['해발고(m)'] = df2['해발고(m)'] / 100 # hm로 변환
    df2['경사(degree)'] = np.tan(np.radians(df2['경사(degree)'])) # tangent로 변환
    df2['방위각(º)'] = np.radians(df2['방위각(º)']) # radian으로 변환
    df2['평균수관밀도(%)'] = df2['평균수관밀도(%)'] / 100 # 소수점 자릿수로 변환
    # ['표본점번호', '수종명', '흉고직경', '수고', '수령', '지하고', '해발고(m)', '경사(degree)', '방위각(º)','평균수관밀도(%)','좌표N', '좌표E']
    df2.columns = ['SampleID', 'Cycle', 'Species', 'Imsang', 'DBH(inch)', 'H(ft)', 'CBH(ft)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'CD(%)', 'Lat', 'Long']
    df2.info

    # 수관높이비율(Crown ratio) 생성
    print("Add new columns: Crown Ratio, Crown Height, Imsang code, Imsang species name...")
    df2.insert(7, 'CH(ft)', (df2['H(ft)'] - df2['CBH(ft)'])) # 수관높이
    df2.insert(6, 'CR', (df2['CH(ft)'] / df2['H(ft)'])) # 수관비율
    # 임상도 기준 수종명 및 수종코드 칼럼 삽입
    df2['I_Species'] = np.full(len(df2), '-99')
    df2['SID'] = np.full(len(df2), -99)
    for key, name_lst in code_species_dict.items(): 
        condition = df2['Species'].isin(name_lst)
        df2.loc[condition, 'I_Species'] = code_name_dict[key]
        df2.loc[condition, 'SID'] = key
        
        condition2 = ((df2['Imsang'] == '활엽수') & (df2['I_Species'] == '-99'))
        df2.loc[condition2, 'I_Species'] = '기타활엽수'
        df2.loc[condition2, 'SID'] = 30
    
        condition3 = ((df2['Imsang'] == '침엽수') & (df2['I_Species'] == '-99'))
        df2.loc[condition3, 'I_Species'] = '기타침엽수'
        df2.loc[condition3, 'SID'] = 10

    print(f"Save the dataframe at {output_name}")
    df_fin = pd.concat([df2[df2.columns[-2:]], df2[df2.columns[:3]], df2[df2.columns[3:15]]], axis=1)
    print(df_fin.shape)
    df_fin = df_fin.dropna()
    df_fin.to_csv(os.path.join(output_name), encoding='cp949', index=False)

    return df_fin

In [ ]:
file_name = r"D:\ForestFire\CBH\data\NFI-Integrated(6-7)-Immok-Filtered.xlsx"
sheet_names = pd.ExcelFile(file_name).sheet_names
print(sheet_names)

In [ ]:
df = pd.read_excel(file_name, sheet_name="NFI6-7")
df_clean = nfiPreprocessing(df, r"D:/ForestFire/CBH/data/NFI6-7_cleaned2.csv")

In [ ]:
df_clean.info()